# 🧠 ReAct: Reasoning + Acting — Interactive Demo

> **Paper**: [Yao et al., ICLR 2023](https://arxiv.org/abs/2210.03629)

This notebook walks through the ReAct implementation step by step:
1. **What is ReAct?** — The paper's core idea
2. **The Wikipedia Tools** — Search and Lookup (no API key needed)
3. **The Prompting Strategy** — Few-shot exemplars
4. **The ReAct Loop** — Full agent execution
5. **Evaluation** — Running on HotpotQA & FEVER

## 1. What is ReAct?

Most LLM agents work in one of two ways:

| Approach | How it works | Problem |
|---|---|---|
| **Chain-of-Thought (CoT)** | Think step-by-step, then answer | Can hallucinate facts mid-chain |
| **Act-only** | Call tools directly | No strategic planning |

**ReAct interleaves both**: the model thinks, acts, observes the result, then thinks again.

```
Thought → Action → Observation → Thought → Action → Observation → ... → Finish
```

The model **thinks out loud, acts, observes, then thinks again** — dramatically reducing
hallucination because it can verify beliefs through real tool calls.

## 2. Setup

First, let's import the components and verify everything loads correctly.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from react_agent import ReactAgent, LLMClient, WikipediaEnv
from react_agent.prompts import build_prompt
from eval.metrics import exact_match, f1_score, accuracy

print('✅ All imports successful!')

## 3. The Wikipedia Environment

The paper uses **three actions** — this minimal tool set is a key insight:

| Action | What it does |
|---|---|
| `Search[entity]` | Returns first 5 sentences of a Wikipedia page |
| `Lookup[keyword]` | Finds next sentence with keyword on current page |
| `Finish[answer]` | Returns the final answer |

> **No API key needed** — these use the public Wikipedia API.

In [ ]:
# Initialize the Wikipedia environment
env = WikipediaEnv()

# ── Search: find a Wikipedia page ──
result = env.search('Python (programming language)')
print('🔍 Search[Python (programming language)]')
print('-' * 50)
print(result[:500])

In [ ]:
# ── Lookup: find specific info on the current page ──
result = env.lookup('Guido')
print('📖 Lookup[Guido]')
print('-' * 50)
print(result)

In [ ]:
# ── Sequential lookups return the NEXT occurrence ──
# This statefulness forces the agent to plan its search strategy
result2 = env.lookup('Guido')
print('📖 Lookup[Guido] (second call)')
print('-' * 50)
print(result2)

### Why this matters

The stateful `lookup` forces the agent to think sequentially:
1. First `search` to find the right page
2. Then `lookup` to find specific details
3. If the first result isn't enough, `lookup` again for the next occurrence

This mimics how humans browse Wikipedia — and it's why ReAct outperforms pure CoT.

## 4. The Prompting Strategy

The few-shot prompt is **the most critical component** of ReAct.
It teaches the model the Thought→Action→Observation pattern through examples.

Let's inspect what a prompt looks like:

In [ ]:
# Build a HotpotQA prompt and inspect it
prompt = build_prompt('hotpotqa', 'What is the capital of France?')

# Show just the structure (first 500 chars + last 200 chars)
print('📝 PROMPT STRUCTURE')
print('=' * 50)
print(prompt[:500])
print('\n... [few-shot examples] ...\n')
print(prompt[-200:])
print('=' * 50)
print(f'\nTotal prompt length: {len(prompt)} characters')

### Key observations about the prompt:

1. **System instruction** — describes the Thought/Action/Observation format
2. **Available tools** — Search, Lookup, Finish
3. **Few-shot examples** — human-written trajectories showing reasoning strategies:
   - *Bridge reasoning*: "Find X, use X to find Y"
   - *Comparison*: "Search both, compare"
   - *Direct*: "Found the answer immediately"
4. **The question** — appended at the end, with space for the trajectory to grow

## 5. The ReAct Agent in Action

Now let's run the full agent. This requires a `GROQ_API_KEY` (free at https://console.groq.com/).

> **If you don't have an API key**, you can still read the output below — the trace shows
> exactly how the agent reasons step by step.

In [ ]:
# ── Multi-hop question (requires GROQ_API_KEY) ──
agent = ReactAgent(task='hotpotqa')
answer, trace = agent.run('What is the capital of the country where the Cheli La pass is located?')

print(f'Answer: {answer}')
print(f'Steps taken: {len(trace)}')

In [ ]:
# ── Pretty-print the reasoning trace ──
agent.print_trace()

### Reading the trace

Each step has three parts:
- 💭 **Thought** — the model's reasoning (decomposition, interpretation, judgment)
- ⚡ **Action** — what tool to call and with what input
- 👁️ **Observation** — the real result from the environment (NOT generated by the LLM)

This is the core insight: *observations come from the real world*, not the model's imagination.

## 6. Anatomy of a Trajectory

Let's inspect the raw trajectory data to understand how the agent "remembers" its reasoning:

In [ ]:
# Inspect each step in the trajectory
for step in trace:
    print(f"\n{'─' * 60}")
    print(f"Step {step['step']}:")
    thought = step['thought']
    print(f"  Thought:  {thought[:100]}..." if len(thought) > 100 else f"  Thought:  {thought}")
    print(f"  Action:   {step['action']}[{step['action_input']}]")
    obs = step['observation']
    print(f"  Observe:  {obs[:100]}..." if len(obs) > 100 else f"  Observe:  {obs}")

## 7. FEVER: Fact Verification

FEVER tests a different skill — instead of extracting an answer, the agent must
**judge** whether a claim is true (SUPPORTS), false (REFUTES), or uncertain (NOT ENOUGH INFO).

In [ ]:
# ── Fact verification (requires GROQ_API_KEY) ──
fever_agent = ReactAgent(task='fever')
answer, trace = fever_agent.run('The Amazon River flows through Africa.')

print(f'Claim: "The Amazon River flows through Africa."')
print(f'Verdict: {answer}')
fever_agent.print_trace()

## 8. Evaluation Metrics

The paper uses standard NLP metrics. Let's verify they work correctly:

In [ ]:
# ── Exact Match — strict comparison after normalization ──
print('Exact Match tests:')
print(f'  "Richard Nixon" vs "Richard Nixon":        {exact_match("Richard Nixon", "Richard Nixon")}')
print(f'  "richard nixon" vs "Richard Nixon":        {exact_match("richard nixon", "Richard Nixon")}')
print(f'  "The Richard Nixon" vs "Richard Nixon":    {exact_match("The Richard Nixon", "Richard Nixon")}')
print(f'  "Barack Obama" vs "Richard Nixon":         {exact_match("Barack Obama", "Richard Nixon")}')

print('\nF1 Score tests (token-level overlap):')
print(f'  "Richard Milhous Nixon" vs "Richard Nixon": {f1_score("Richard Milhous Nixon", "Richard Nixon"):.2f}')
print(f'  "The Great Wall" vs "Great Wall of China":  {f1_score("The Great Wall", "Great Wall of China"):.2f}')

print('\nAccuracy tests (for FEVER):')
print(f'  "SUPPORTS" vs "SUPPORTS":    {accuracy("SUPPORTS", "SUPPORTS")}')
print(f'  "supports" vs "SUPPORTS":    {accuracy("supports", "SUPPORTS")}')
print(f'  "REFUTES" vs "SUPPORTS":     {accuracy("REFUTES", "SUPPORTS")}')

## 9. Running Full Evaluations

To evaluate on the curated HotpotQA and FEVER subsets:

```bash
# From the project root
python -m eval.run_hotpotqa --n 5     # Multi-hop QA
python -m eval.run_fever --n 5        # Fact verification
```

Or run from this notebook:

In [ ]:
# Uncomment to run evaluation (uses API calls)
# from eval.run_hotpotqa import run_evaluation as run_hotpotqa
# results = run_hotpotqa(n=3, verbose=True)

## 10. Key Takeaways

### What makes ReAct work:

1. **The stop sequence** (`Observation:`) prevents the LLM from hallucinating search results
2. **Few-shot prompts** teach reasoning patterns, not just output format
3. **Interleaving** thought and action provides both planning AND grounding
4. **The simplicity** — just 3 tools and a prompt — is itself the insight

### Component summary:

| Component | File | Key idea |
|---|---|---|
| ReAct loop | `react_agent/agent.py` | Thought→Action→Observation cycle |
| LLM wrapper | `react_agent/llm.py` | Stop at `Observation:` to prevent hallucination |
| Tools | `react_agent/tools.py` | Search + Lookup = minimal but sufficient |
| Prompts | `react_agent/prompts.py` | Few-shot exemplars from the paper |
| Metrics | `eval/metrics.py` | Standard NLP: EM, F1, accuracy |
| Eval | `eval/run_*.py` | HotpotQA + FEVER benchmarks |

### The big picture:

> *"ReAct shows that you don't need complex agent frameworks. The interleaving of
> reasoning and acting — with proper prompting — is itself the fundamental contribution."*